In [1]:
import pandas as pd
import duckdb

# Summarize `sales.csv` into `monthly-sales.csv`

## Load sales.csv 

- This test data generated by script Grok made.
- 2 suppliers with 2 licensees each, plus another licensee, who buys from both suppliers.
- Date range 3/2023 to 1/2026

In [2]:
sales = pd.read_csv("./csv/sales-data-generated-by-grok.csv", parse_dates=['DATE'])
sales['AMT'] = sales['AMT'].astype('int64')
sales['QTY'] = sales['QTY'].astype('int64')
sales

,DATE,SUPPLIER,LICENSEE,PO,PART,QTY,AMT
0,2023-03-21,nippon-metal,ninja-roofing,PO0001,gold-decoration,143,57300
1,2023-03-21,nippon-metal,ninja-roofing,PO0001,gold-clip,117,55600
2,2023-03-21,nippon-metal,ninja-roofing,PO0001,tin-decoration,246,21400
3,2023-03-21,nippon-metal,ninja-roofing,PO0001,glass-decoration,236,213500
4,2023-03-21,nippon-metal,ninja-roofing,PO0001,glass-clip,85,101500
...,...,...,...,...,...,...,...
798,2026-01-26,us-steel,global-roof,PO0167,roof-polish,234,60000
799,2026-01-26,us-steel,global-roof,PO0167,gold-clip,231,136600
800,2026-01-26,us-steel,global-roof,PO0167,tin-decoration,54,4500
801,2026-01-26,us-steel,global-roof,PO0167,glass-clip,165,191300


## Group sales by month, supplier, licensee, and part.  
## Calculate monthly and YTD totals.

In [3]:
duckdb.query("DROP VIEW IF EXISTS monthly_sales");

duckdb.query("""
CREATE VIEW monthly_sales AS
SELECT 
    DATE_TRUNC('month', date) AS month,
    supplier,
    licensee,
    part,
    CAST(SUM(qty) AS INT64) AS qty,
    CAST(SUM(amt) AS INT64) AS month_amt,
    CAST(
        SUM(month_amt) OVER (
            PARTITION BY EXTRACT(year FROM month), supplier, licensee, part 
            ORDER BY month
        ) AS INT64
    ) AS ytd_amt
FROM sales
GROUP BY month, supplier, licensee, part
ORDER BY month, supplier, licensee, part;
""")

ytd_df = duckdb.query("SELECT * FROM monthly_sales").df()

ytd_df

,month,SUPPLIER,LICENSEE,PART,qty,month_amt,ytd_amt
0,2023-03-01,nippon-metal,global-roof,glass-clip,194,242300,242300
1,2023-03-01,nippon-metal,global-roof,glass-decoration,189,139800,139800
2,2023-03-01,nippon-metal,global-roof,gold-clip,220,105400,105400
3,2023-03-01,nippon-metal,global-roof,gold-decoration,131,67500,67500
4,2023-03-01,nippon-metal,global-roof,roof-polish,139,29500,29500
...,...,...,...,...,...,...,...
798,2026-01-01,us-steel,zztop-roof,glass-decoration,107,92400,92400
799,2026-01-01,us-steel,zztop-roof,gold-clip,106,50200,50200
800,2026-01-01,us-steel,zztop-roof,gold-decoration,127,52600,52600
801,2026-01-01,us-steel,zztop-roof,tin-clip,80,7700,7700


\
*save to csv*

In [5]:
ytd_df.to_csv('./csv/sales-data-summary-by-month.csv', index=False)